## Phần 1: Khởi tạo môi trường và khám phá dữ liệu

### Bước 1: Khởi tạo SparkSession và nạp dữ liệu đã làm sạch từ HDFS

In [1]:
import os
import sys
import math
import findspark
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
findspark.init()

from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("Train_Model_Du_Doan_Mua_Hang") \
    .master("spark://26.37.93.102:7077") \
    .config("spark.executor.memory", "10g") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
get_prob1 = F.udf(lambda v: float(v[1]), DoubleType())

print("Đã tạo thành công kết nối SparkSession!")
df = spark.read.parquet("hdfs://master:9000/data/test_cleaned.parquet")
print(f"Tổng số dòng dữ liệu sẵn sàng cho mô hình: {df.count():,}")

Đã tạo thành công kết nối SparkSession!
Tổng số dòng dữ liệu sẵn sàng cho mô hình: 2,711,122


### Bước 2: Khám phá dữ liệu và kiểm tra phân phối nhãn

In [2]:
print("-" * 60)
print("THÔNG TIN SCHEMA")
print("-" * 60)
df.printSchema()

print("\n" + "-" * 60)
print("PHÂN PHỐI LOẠI SỰ KIỆN (event_type)")
print("-" * 60)
df.groupBy("event_type").count().orderBy(F.desc("count")).show()

print("\n" + "-" * 60)
print("PHÂN PHỐI NHÃN MỤC TIÊU (target)")
print("-" * 60)
df.groupBy("target").count().orderBy("target").show()

print("\n" + "-" * 60)
print("THỐNG KÊ MÔ TẢ CỘT SỐ")
print("-" * 60)
df.select("price", "ts_hour", "ts_weekday").describe().show()

------------------------------------------------------------
THÔNG TIN SCHEMA
------------------------------------------------------------
root
 |-- product_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)
 |-- target: long (nullable = true)
 |-- cat_0: string (nullable = true)
 |-- cat_1: string (nullable = true)
 |-- cat_2: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- ts_hour: short (nullable = true)
 |-- ts_minute: short (nullable = true)
 |-- ts_weekday: short (nullable = true)
 |-- ts_day: short (nullable = true)
 |-- ts_month: short (nullable = true)
 |-- ts_year: short (nullable = true)


------------------------------------------------------------
PHÂN PHỐI LOẠI SỰ KIỆN (event_type)
--------------------------------------------------

---
## Phần 2: Tiền xử lý đặc trưng đầu vào

### Bước 3: Lọc dữ liệu và tạo nhãn

In [3]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# 1. Lọc dữ liệu
df_valid = df.filter(
    (F.col("price") > 0) &
    F.col("brand").isNotNull() &
    F.col("cat_0").isNotNull() &
    F.col("user_session").isNotNull()
)

# 2. Xây dựng Window để nhìn vào toàn bộ 1 phiên làm việc đối với 1 sản phẩm
window_session_prod = Window.partitionBy("user_session", "product_id")

# 3. Tạo nhãn
df_target = df_valid.withColumn(
    "is_purchased",
    F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).over(window_session_prod)
)

# 4. CHỈ LẤY CÁC SỰ KIỆN 'CART' ĐỂ DỰ ĐOÁN TƯƠNG LAI
df_cart_only = df_target.filter(F.col("event_type") == "cart")
df_model_base = df_cart_only.dropDuplicates(["user_session", "product_id"])
df_labeled = df_model_base.withColumnRenamed("is_purchased", "label")

# Tính toán thống kê lớp dữ liệu
total = df_labeled.count()
pos = df_labeled.filter(F.col("label") == 1).count()
neg = total - pos

print("-" * 60)
print("THỐNG KÊ SAU KHI ĐỊNH NGHĨA LẠI NHÃN")
print("-" * 60)
print(f"Tổng số sự kiện thêm vào giỏ (Cart) : {total:,}")
print(f"Khách chốt đơn sau khi Cart (Label=1): {pos:,} ({(pos/total)*100:.2f}%)")
print(f"Khách bỏ giỏ hàng (Label=0)          : {neg:,} ({(neg/total)*100:.2f}%)")
if pos > 0:
    print(f"Tỷ lệ chênh lệch lớp                 : 1:{neg/pos:.2f}")

------------------------------------------------------------
THỐNG KÊ SAU KHI ĐỊNH NGHĨA LẠI NHÃN
------------------------------------------------------------
Tổng số sự kiện thêm vào giỏ (Cart) : 1,240,432
Khách chốt đơn sau khi Cart (Label=1): 4,173 (0.34%)
Khách bỏ giỏ hàng (Label=0)          : 1,236,259 (99.66%)
Tỷ lệ chênh lệch lớp                 : 1:296.25


### Bước 4: Tạo đặc trưng mới

In [4]:
df_feat = df_labeled \
    .withColumn(
        "price_log",
        F.log1p(F.col("price"))
    ) \
    .withColumn(
        "is_golden_hour",
        F.when(
            (F.col("ts_hour").between(10, 12)) | (F.col("ts_hour").between(20, 22)), 1
        ).otherwise(0).cast(IntegerType())
    ) \
    .withColumn(
        "session_part",
        F.when(F.col("ts_hour").between(0, 5), "night")
         .when(F.col("ts_hour").between(6, 11), "morning")
         .when(F.col("ts_hour").between(12, 17), "afternoon")
         .otherwise("evening")
    )

numeric_cols = ["price_log", "is_golden_hour", "ts_weekday"]

print("Các đặc trưng sẵn sàng đưa vào mô hình:")
df_feat.select(
    "price_log", "session_part", "ts_weekday",
    "is_golden_hour", "brand", "cat_0", "label"
).show(5, truncate=False)

Các đặc trưng sẵn sàng đưa vào mô hình:
+------------------+------------+----------+--------------+-------+------------+-----+
|price_log         |session_part|ts_weekday|is_golden_hour|brand  |cat_0       |label|
+------------------+------------+----------+--------------+-------+------------+-----+
|4.8630629154809615|night       |1         |0             |samsung|construction|0    |
|6.09807428216624  |morning     |4         |0             |samsung|construction|0    |
|6.732043809765473 |afternoon   |5         |0             |apple  |construction|0    |
|5.376157661959246 |night       |1         |0             |kk     |appliances  |0    |
|4.590361057945231 |afternoon   |2         |0             |sony   |sport       |0    |
+------------------+------------+----------+--------------+-------+------------+-----+
only showing top 5 rows



### Bước 5: Xây dựng Spark ML Pipeline

In [5]:
cat_cols     = ["brand", "cat_0", "session_part"]
numeric_cols = ["price_log", "is_golden_hour", "ts_hour", "ts_weekday"]

indexed_cols = [c + "_idx" for c in cat_cols]
encoded_cols = [c + "_ohe" for c in cat_cols]

# StringIndexer: chuyển chuỗi thành số nguyên
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="skip")
    for c in cat_cols
]

# OneHotEncoder: chuyển số nguyên thành vector nhị phân
encoder = OneHotEncoder(
    inputCols=indexed_cols,
    outputCols=encoded_cols,
    dropLast=True
)

# VectorAssembler: gom tất cả đặc trưng thành 1 vector
all_feature_cols = numeric_cols + encoded_cols
assembler = VectorAssembler(
    inputCols=all_feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

# StandardScaler: chuẩn hóa theo độ lệch chuẩn
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

print("Đã định nghĩa các bước tiền xử lý trong Pipeline:")
print(f"   Categorical (OHE): {cat_cols}")
print(f"   Numeric           : {numeric_cols}")
print(f"   Tổng số lượng đặc trưng: {len(all_feature_cols)} (trước khi OneHot expand)")

Đã định nghĩa các bước tiền xử lý trong Pipeline:
   Categorical (OHE): ['brand', 'cat_0', 'session_part']
   Numeric           : ['price_log', 'is_golden_hour', 'ts_hour', 'ts_weekday']
   Tổng số lượng đặc trưng: 7 (trước khi OneHot expand)


### Bước 6: Phân chia tập Train/ Test & xử lý mất cân bằng lớp

In [6]:
train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)

# Kiểm tra phân phối nhãn sau split
train_count = train_df.count()
test_count  = test_df.count()
pos_count   = train_df.filter(F.col("label") == 1).count()
neg_count   = train_count - pos_count
balance_ratio = neg_count / pos_count if pos_count > 0 else 1.0

pos_test = test_df.filter(F.col("label") == 1).count()

print(f"Kích thước tập Train : {train_count:,} dòng")
print(f"  Label=1 trong Train: {pos_count:,} ({pos_count/train_count*100:.2f}%)")
print(f"  Label=0 trong Train: {neg_count:,} ({neg_count/train_count*100:.2f}%)")
print(f"Kích thước tập Test  : {test_count:,} dòng")
print(f"  Label=1 trong Test : {pos_test:,} ({pos_test/test_count*100:.2f}%)")
print(f"Hệ số cân bằng lớp (neg/pos): {balance_ratio:.2f} → dùng làm trọng số classWeight")

# Gắn cột classWeight để bù đắp mất cân bằng
train_weighted = train_df.withColumn(
    "classWeight",
    F.when(F.col("label") == 1, balance_ratio).otherwise(1.0)
)

Kích thước tập Train : 992,251 dòng
  Label=1 trong Train: 3,376 (0.34%)
  Label=0 trong Train: 988,875 (99.66%)
Kích thước tập Test  : 248,181 dòng
  Label=1 trong Test : 797 (0.32%)
Hệ số cân bằng lớp (neg/pos): 292.91 → dùng làm trọng số classWeight


---
## Phần 2: Đào tạo và Đánh giá Mô hình

### Bước 7: Huấn luyện mô hình Logistic Regression

In [7]:
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier, LogisticRegression
lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.1,
    weightCol="classWeight",
    family="binomial"
)

pipeline_lr = Pipeline(stages=indexers + [encoder, assembler, scaler, lr])

print("⏳ Đang huấn luyện Logistic Regression...")
model_lr = pipeline_lr.fit(train_weighted)
print("✅ Huấn luyện Logistic Regression hoàn tất!")

⏳ Đang huấn luyện Logistic Regression...
✅ Huấn luyện Logistic Regression hoàn tất!


### Bước 8: Huấn luyện mô hình Random Forest Classifier

In [8]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=8,
    minInstancesPerNode=10,
    featureSubsetStrategy="sqrt",
    weightCol="classWeight",
    seed=42
)

# Pipeline hoàn chỉnh: tiền xử lý + mô hình
pipeline_rf = Pipeline(stages=indexers + [encoder, assembler, scaler, rf])

print("⏳ Đang huấn luyện Random Forest...")
model_rf = pipeline_rf.fit(train_weighted)
print("✅ Huấn luyện Random Forest hoàn tất!")

⏳ Đang huấn luyện Random Forest...
✅ Huấn luyện Random Forest hoàn tất!


### Bước 9: Huấn luyện mô hình Gradient-Boosted Trees (GBT)

In [9]:
train_weighted.persist()
train_weighted.count()

gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxIter=40,
    maxDepth=4,
    stepSize=0.1,
    subsamplingRate=0.8,
    featureSubsetStrategy="sqrt",
    weightCol="classWeight",
    seed=42
)

# Thiết lập Pipeline tiền xử lý và huấn luyện
pipeline_gbt = Pipeline(stages=indexers + [encoder, assembler, scaler, gbt])

print("⏳ Đang huấn luyện GBT...")
model_gbt = pipeline_gbt.fit(train_weighted)
print("✅ Huấn luyện GBT hoàn tất!")

train_weighted.unpersist()

⏳ Đang huấn luyện GBT...
✅ Huấn luyện GBT hoàn tất!


DataFrame[product_id: string, event_time: string, event_type: string, brand: string, price: double, user_id: string, user_session: string, target: bigint, cat_0: string, cat_1: string, cat_2: string, timestamp: timestamp, ts_hour: smallint, ts_minute: smallint, ts_weekday: smallint, ts_day: smallint, ts_month: smallint, ts_year: smallint, label: int, price_log: double, is_golden_hour: int, session_part: string, classWeight: double]

### Bước 10: Dự đoán và thu thập kết quả

In [10]:
# [FIX] Thêm pred_lr — trước đây LR được train ở bước 7 nhưng không bao giờ dùng để predict
pred_lr  = model_lr.transform(test_df)
pred_rf  = model_rf.transform(test_df)
pred_gbt = model_gbt.transform(test_df)

print("Mẫu kết quả dự đoán (Random Forest):")
pred_rf.select("label", "prediction", "probability").show(5)

Mẫu kết quả dự đoán (Random Forest):
+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    0|       1.0|[0.48776593235318...|
|    0|       0.0|[0.52597971095504...|
|    0|       0.0|[0.52321395340980...|
|    0|       1.0|[0.49998850482071...|
|    0|       0.0|[0.52049867237001...|
+-----+----------+--------------------+
only showing top 5 rows



### Bước 11: Đánh giá mô hình - ROC-AUC, F1, Precision, Recall, MCC

In [11]:
def evaluate_model(predictions, model_name):
    """Tính đầy đủ các chỉ số đánh giá cho bài toán phân loại nhị phân."""
    # ROC-AUC
    bin_eval = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    )
    roc_auc = bin_eval.evaluate(predictions)

    pr_auc = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
    ).evaluate(predictions)

    # F1, Precision, Recall, Accuracy
    mc_eval = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction"
    )
    f1        = mc_eval.evaluate(predictions, {mc_eval.metricName: "f1"})
    precision = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedPrecision"})
    recall    = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedRecall"})
    accuracy  = mc_eval.evaluate(predictions, {mc_eval.metricName: "accuracy"})

    # Confusion Matrix → MCC
    cm = predictions.groupBy("label", "prediction").count().collect()
    cm_dict = {(int(r["label"]), int(r["prediction"])): r["count"] for r in cm}
    TP = cm_dict.get((1, 1), 0)
    TN = cm_dict.get((0, 0), 0)
    FP = cm_dict.get((0, 1), 0)
    FN = cm_dict.get((1, 0), 0)

    denom = math.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
    mcc   = (TP*TN - FP*FN) / denom if denom > 0 else 0.0

    print(f"\n{'='*55}")
    print(f"  KẾT QUẢ ĐÁNH GIÁ: {model_name}")
    print(f"{'='*55}")
    print(f"  ROC-AUC   : {roc_auc:.4f}")
    print(f"  PR-AUC    : {pr_auc:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  Accuracy  : {accuracy:.4f}")
    print(f"  MCC       : {mcc:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"  {'':12} Pred=0    Pred=1")
    print(f"  {'Actual=0':12} {TN:>8,}  {FP:>8,}")
    print(f"  {'Actual=1':12} {FN:>8,}  {TP:>8,}")
    print(f"{'='*55}")

    return {
        "model": model_name, "roc_auc": roc_auc, "pr_auc": pr_auc,
        "f1": f1, "precision": precision, "recall": recall,
        "accuracy": accuracy, "mcc": mcc,
        "TP": TP, "TN": TN, "FP": FP, "FN": FN
    }

metrics_lr  = evaluate_model(pred_lr,  "Logistic Regression")
metrics_rf  = evaluate_model(pred_rf,  "Random Forest")
metrics_gbt = evaluate_model(pred_gbt, "Gradient-Boosted Trees")


  KẾT QUẢ ĐÁNH GIÁ: Logistic Regression
  ROC-AUC   : 0.5835
  PR-AUC    : 0.0044
  F1-Score  : 0.7138
  Precision : 0.9943
  Recall    : 0.5585
  Accuracy  : 0.5585
  MCC       : 0.0137

  Confusion Matrix:
               Pred=0    Pred=1
  Actual=0      138,108   109,170
  Actual=1          349       448

  KẾT QUẢ ĐÁNH GIÁ: Random Forest
  ROC-AUC   : 0.6049
  PR-AUC    : 0.0048
  F1-Score  : 0.7446
  Precision : 0.9943
  Recall    : 0.5968
  Accuracy  : 0.5968
  MCC       : 0.0157

  Confusion Matrix:
               Pred=0    Pred=1
  Actual=0      147,622    99,656
  Actual=1          367       430

  KẾT QUẢ ĐÁNH GIÁ: Gradient-Boosted Trees
  ROC-AUC   : 0.6124
  PR-AUC    : 0.0049
  F1-Score  : 0.7310
  Precision : 0.9944
  Recall    : 0.5797
  Accuracy  : 0.5797
  MCC       : 0.0165

  Confusion Matrix:
               Pred=0    Pred=1
  Actual=0      143,402   103,950
  Actual=1          347       450


### Bước 12: So sánh hai mô hình

In [12]:
comparison_data = [
    Row(Model=m["model"], ROC_AUC=round(m["roc_auc"],4),
        PR_AUC=round(m["pr_auc"],4), F1=round(m["f1"],4),
        Precision=round(m["precision"],4), Recall=round(m["recall"],4),
        MCC=round(m["mcc"],4), Accuracy=round(m["accuracy"],4))
    for m in [metrics_lr, metrics_rf, metrics_gbt]
]

comparison_df = spark.createDataFrame(comparison_data)
print("\n📊 SO SÁNH BA MÔ HÌNH:")
comparison_df.show(truncate=False)

# Chọn mô hình tốt nhất dựa trên F1-Score
best = max([metrics_lr, metrics_rf, metrics_gbt], key=lambda x: x["f1"])
print(f"🏆 Mô hình tốt nhất theo F1-Score: {best['model']}  (F1={best['f1']:.4f}, MCC={best['mcc']:.4f})")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\DELL\\AppData\\Local\\Temp\\spark-25855acd-a04b-45c3-acde-e0d84084347a\\pyspark-95c8b487-f301-4eb7-82e3-f153331426c8\\tmpow2p1ibq'

### Bước 13: Feature Importance - Random Forest

In [ ]:
rf_model = model_rf.stages[-1]
attrs = pred_rf.schema["features"].metadata["ml_attr"]["attrs"]
feature_names = [
    name for _, name in sorted(
        [(a["idx"], a["name"]) for a in attrs.get("binary", []) + attrs.get("numeric", [])],
        key=lambda x: x[0]
    )
]

feature_importances = rf_model.featureImportances.toArray()
fi_pairs = sorted(zip(feature_names, feature_importances), key=lambda x: -x[1])

print("\n TOP 15 ĐẶC TRƯNG QUAN TRỌNG NHẤT (Random Forest):")
print(f"{'Rank':>4}  {'Feature':<30}  {'Importance':>12}")
print("-" * 52)
for rank, (feat, imp) in enumerate(fi_pairs[:15], 1):
    bar = "█" * int(imp * 200)
    print(f"{rank:>4}  {feat:<30}  {imp:>10.4f}  {bar}")

---
## Phần 3: Trực quan hóa mô hình

### Bước 14: Thu thập dữ liệu để vẽ biểu đồ

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_curve, precision_recall_curve
import numpy as np

SAMPLE_SIZE = 50_000

def collect_proba(spark_pred_df, sample_size=SAMPLE_SIZE):
    """Collect label + probability score về Pandas."""
    total = spark_pred_df.count()
    df_small = spark_pred_df.select(
        F.col("label"),
        get_prob1(F.col("probability")).alias("prob_1")
    )
    if total > sample_size:
        df_small = df_small.sample(fraction=sample_size / total, seed=42)
    return df_small.toPandas()

pdf_lr  = collect_proba(pred_lr)
pdf_rf  = collect_proba(pred_rf)
pdf_gbt = collect_proba(pred_gbt)

# Feature importance pandas
fi_df = pd.DataFrame(fi_pairs[:15], columns=["Feature", "Importance"])

print(f"Đã collect {len(pdf_lr):,} mẫu LR, {len(pdf_rf):,} mẫu RF và {len(pdf_gbt):,} mẫu GBT về driver.")

### Bước 15: Vẽ toàn bộ biểu đồ đánh giá mô hình

In [ ]:
fig = plt.figure(figsize=(20, 22))
fig.suptitle(
    "Báo cáo Đánh giá Mô hình: Dự đoán Khả năng Mua hàng",
    fontsize=16, fontweight="bold", y=0.98
)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

COLORS = {"LR": "#4CAF50", "RF": "#2196F3", "GBT": "#FF5722", "pos": "#4CAF50", "neg": "#F44336"}

# 1. ROC CURVE
ax1 = fig.add_subplot(gs[0, 0])
for pdf, label, color in [
    (pdf_lr,  f"LR  (AUC={metrics_lr['roc_auc']:.3f})",  COLORS["LR"]),
    (pdf_rf,  f"RF  (AUC={metrics_rf['roc_auc']:.3f})",  COLORS["RF"]),
    (pdf_gbt, f"GBT (AUC={metrics_gbt['roc_auc']:.3f})", COLORS["GBT"])
]:
    fpr, tpr, _ = roc_curve(pdf["label"], pdf["prob_1"])
    ax1.plot(fpr, tpr, color=color, lw=2, label=label)
ax1.plot([0,1],[0,1], "k--", lw=1, label="Random")
ax1.set_xlabel("False Positive Rate"); ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curve", fontweight="bold")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# 2. PRECISION-RECALL CURVE
ax2 = fig.add_subplot(gs[0, 1])
for pdf, label, color in [
    (pdf_lr,  f"LR  (PR-AUC={metrics_lr['pr_auc']:.3f})",  COLORS["LR"]),
    (pdf_rf,  f"RF  (PR-AUC={metrics_rf['pr_auc']:.3f})",  COLORS["RF"]),
    (pdf_gbt, f"GBT (PR-AUC={metrics_gbt['pr_auc']:.3f})", COLORS["GBT"])
]:
    prec, rec, _ = precision_recall_curve(pdf["label"], pdf["prob_1"])
    ax2.plot(rec, prec, color=color, lw=2, label=label)
baseline = pdf_rf["label"].mean()
ax2.axhline(baseline, color="gray", ls="--", lw=1, label=f"Baseline ({baseline:.3f})")
ax2.set_xlabel("Recall"); ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curve", fontweight="bold")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# 3. BIỂU ĐỒ SO SÁNH CHỈ SỐ ĐÁNH GIÁ (3 mô hình)
ax3 = fig.add_subplot(gs[0, 2])
metrics_keys = ["ROC-AUC", "PR-AUC", "F1", "Precision", "Recall", "MCC"]
vals_lr  = [metrics_lr["roc_auc"],  metrics_lr["pr_auc"],  metrics_lr["f1"],
            metrics_lr["precision"],  metrics_lr["recall"],  max(metrics_lr["mcc"],0)]
vals_rf  = [metrics_rf["roc_auc"],  metrics_rf["pr_auc"],  metrics_rf["f1"],
            metrics_rf["precision"],  metrics_rf["recall"],  max(metrics_rf["mcc"],0)]
vals_gbt = [metrics_gbt["roc_auc"], metrics_gbt["pr_auc"], metrics_gbt["f1"],
            metrics_gbt["precision"], metrics_gbt["recall"], max(metrics_gbt["mcc"],0)]

x = np.arange(len(metrics_keys))
w = 0.25
ax3.bar(x - w, vals_lr,  w, label="LR",  color=COLORS["LR"],  alpha=0.85)
ax3.bar(x,     vals_rf,  w, label="RF",  color=COLORS["RF"],  alpha=0.85)
ax3.bar(x + w, vals_gbt, w, label="GBT", color=COLORS["GBT"], alpha=0.85)
ax3.set_xticks(x); ax3.set_xticklabels(metrics_keys, rotation=20, ha="right", fontsize=9)
ax3.set_ylim(0, 1.15); ax3.set_ylabel("Score")
ax3.set_title("So sánh chỉ số 3 mô hình", fontweight="bold")
ax3.legend(fontsize=9); ax3.grid(axis="y", alpha=0.3)

# 4. CONFUSION MATRIX — RANDOM FOREST
ax4 = fig.add_subplot(gs[1, 0])
cm_rf   = np.array([[metrics_rf["TN"],  metrics_rf["FP"]],
                    [metrics_rf["FN"],  metrics_rf["TP"]]])
cm_norm = cm_rf / cm_rf.sum(axis=1, keepdims=True)
im = ax4.imshow(cm_norm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax4.text(j, i, f"{cm_rf[i,j]:,}\n({cm_norm[i,j]:.1%})",
                 ha="center", va="center", fontsize=10, fontweight="bold",
                 color="white" if cm_norm[i,j] > 0.6 else "black")
ax4.set_xticks([0,1]); ax4.set_yticks([0,1])
ax4.set_xticklabels(["Pred: No Buy","Pred: Buy"])
ax4.set_yticklabels(["Actual: No Buy","Actual: Buy"])
ax4.set_title("Confusion Matrix — Random Forest", fontweight="bold")
plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

# 5. CONFUSION MATRIX — GBT
ax5 = fig.add_subplot(gs[1, 1])
cm_gbt  = np.array([[metrics_gbt["TN"], metrics_gbt["FP"]],
                    [metrics_gbt["FN"], metrics_gbt["TP"]]])
cm_norm2 = cm_gbt / cm_gbt.sum(axis=1, keepdims=True)
im2 = ax5.imshow(cm_norm2, cmap="Oranges")
for i in range(2):
    for j in range(2):
        ax5.text(j, i, f"{cm_gbt[i,j]:,}\n({cm_norm2[i,j]:.1%})",
                 ha="center", va="center", fontsize=10, fontweight="bold",
                 color="white" if cm_norm2[i,j] > 0.6 else "black")
ax5.set_xticks([0,1]); ax5.set_yticks([0,1])
ax5.set_xticklabels(["Pred: No Buy","Pred: Buy"])
ax5.set_yticklabels(["Actual: No Buy","Actual: Buy"])
ax5.set_title("Confusion Matrix — GBT", fontweight="bold")
plt.colorbar(im2, ax=ax5, fraction=0.046, pad=0.04)

# 6. PHÂN PHỐI XÁC SUẤT DỰ ĐOÁN (RF)
ax6 = fig.add_subplot(gs[1, 2])
for lbl, color, name in [(0, COLORS["neg"], "No Buy (0)"), (1, COLORS["pos"], "Buy (1)")]:
    subset = pdf_rf[pdf_rf["label"] == lbl]["prob_1"]
    ax6.hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)
ax6.axvline(0.5, color="black", ls="--", lw=1.5, label="Ngưỡng 0.5")
ax6.set_xlabel("P(purchase)"); ax6.set_ylabel("Mật độ")
ax6.set_title("Phân phối xác suất dự đoán (RF)", fontweight="bold")
ax6.legend(fontsize=9); ax6.grid(alpha=0.3)

# 7. FEATURE IMPORTANCE
ax7 = fig.add_subplot(gs[2, :])
colors_fi = [COLORS["RF"] if "_ohe" not in f else "#9C27B0" for f in fi_df["Feature"]]
bars = ax7.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
                color=colors_fi[::-1], alpha=0.85)
for bar, val in zip(bars, fi_df["Importance"][::-1]):
    ax7.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", fontsize=8)
patch_num = mpatches.Patch(color=COLORS["RF"],  label="Numeric feature")
patch_cat = mpatches.Patch(color="#9C27B0",     label="Categorical (OHE)")
ax7.legend(handles=[patch_num, patch_cat], fontsize=9)
ax7.set_xlabel("Feature Importance (Gini)")
ax7.set_title("Top 15 Feature Importances — Random Forest", fontweight="bold")
ax7.grid(axis="x", alpha=0.3)

plt.savefig("/home/claude/model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Đã lưu biểu đồ: model_evaluation.png")

### Bước 16: Phân tích xu hướng xác suất mua theo giờ & ngày

In [ ]:
hourly_df = pred_rf \
    .withColumn("prob_purchase", get_prob1(F.col("probability"))) \
    .groupBy("ts_hour") \
    .agg(
        F.mean("prob_purchase").alias("avg_prob"),
        F.mean("label").alias("actual_rate"),
        F.count("*").alias("n")
    ) \
    .orderBy("ts_hour") \
    .toPandas()

weekday_labels = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
daily_df = pred_rf \
    .withColumn("prob_purchase", get_prob1(F.col("probability"))) \
    .groupBy("ts_weekday") \
    .agg(
        F.mean("prob_purchase").alias("avg_prob"),
        F.mean("label").alias("actual_rate")
    ) \
    .orderBy("ts_weekday") \
    .toPandas()
daily_df["day_label"] = daily_df["ts_weekday"].apply(
    lambda x: weekday_labels[int(x)] if int(x) < 7 else str(x)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Phân tích Xu hướng Xác suất Mua hàng theo Thời gian",
             fontsize=13, fontweight="bold")

# --- Biểu đồ theo giờ ---
ax = axes[0]
ax.fill_between(hourly_df["ts_hour"], hourly_df["avg_prob"], alpha=0.3, color=COLORS["RF"])
ax.plot(hourly_df["ts_hour"], hourly_df["avg_prob"],
        color=COLORS["RF"], lw=2, marker="o", ms=5, label="Dự đoán TB")
ax.plot(hourly_df["ts_hour"], hourly_df["actual_rate"],
        color=COLORS["GBT"], lw=1.5, ls="--", marker="s", ms=4, label="Tỷ lệ thực tế")
for span in [(10, 12), (20, 22)]:
    ax.axvspan(span[0], span[1], alpha=0.12, color="gold")
ax.set_xlabel("Giờ trong ngày"); ax.set_ylabel("Xác suất mua")
ax.set_title("Theo giờ (vùng vàng = giờ vàng)", fontweight="bold")
ax.set_xticks(range(0, 24, 2)); ax.legend(); ax.grid(alpha=0.3)

# --- Biểu đồ theo ngày ---
ax = axes[1]
bar_colors = [COLORS["GBT"] if d in ["Sat","Sun"] else COLORS["RF"]
              for d in daily_df["day_label"]]
ax.bar(daily_df["day_label"], daily_df["avg_prob"], color=bar_colors, alpha=0.8)

line_actual, = ax.plot(daily_df["day_label"], daily_df["actual_rate"],
        color="black", lw=1.5, marker="D", ms=6, zorder=5)

for i, (d, v) in enumerate(zip(daily_df["day_label"], daily_df["avg_prob"])):
    ax.text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=8)

wd_patch = mpatches.Patch(color=COLORS["RF"],  label="Ngày thường")
we_patch = mpatches.Patch(color=COLORS["GBT"], label="Cuối tuần")
ax.legend(handles=[wd_patch, we_patch, line_actual],
          labels=["Ngày thường", "Cuối tuần", "Tỷ lệ thực tế"], fontsize=9)
ax.set_xlabel("Ngày trong tuần"); ax.set_ylabel("Xác suất mua")
ax.set_title("Theo ngày trong tuần", fontweight="bold")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("/home/claude/time_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Đã lưu biểu đồ: time_analysis.png")

### Bước 17: Lưu mô hình tốt nhất lên HDFS

In [ ]:
best_model_name = best["model"]
if best_model_name == "Logistic Regression":
    best_pipeline = model_lr
elif best_model_name == "Random Forest":
    best_pipeline = model_rf
else:
    best_pipeline = model_gbt

save_path = f"hdfs://master:9000/models/purchase_prediction_{best_model_name.replace(' ','_').lower()}"

best_pipeline.write().overwrite().save(save_path)
print(f"✅ Đã lưu mô hình '{best_model_name}' lên HDFS:")
print(f"   {save_path}")

### Bước 18: Ứng dụng thực tiễn - Phân khúc khách hàng có xác suất mua cao

In [ ]:
THRESHOLD_HIGH   = 0.7
THRESHOLD_MEDIUM = 0.4

pred_application = pred_rf \
    .withColumn("prob_purchase", get_prob1(F.col("probability"))) \
    .filter(F.col("label") == 0)

segments = pred_application \
    .withColumn(
        "segment",
        F.when(F.col("prob_purchase") >= THRESHOLD_HIGH,   "Ưu tiên cao — Gửi mã FREESHIP ngay")
         .when(F.col("prob_purchase") >= THRESHOLD_MEDIUM, "Ưu tiên TB  — Nhắc nhở giỏ hàng")
         .otherwise("Ưu tiên thấp — Chiến dịch retarget")
    )

print("\n PHÂN KHÚC KHÁCH HÀNG BỎ GIỎ THEO DỰ ĐOÁN CỦA MÔ HÌNH:")
segments.groupBy("segment") \
    .agg(
        F.count("*").alias("so_khach"),
        F.round(F.mean("prob_purchase"), 4).alias("xac_suat_tb"),
        F.round(F.mean("price"), 2).alias("gia_tb")
    ) \
    .orderBy(F.desc("xac_suat_tb")) \
    .show(truncate=False)

high_value = segments.filter(F.col("prob_purchase") >= THRESHOLD_HIGH).count()
print(f"\n Insight: Có {high_value:,} khách hàng cần can thiệp ngay (Xác suất chốt đơn ≥ {THRESHOLD_HIGH}).")
print("   → Gửi push notification 'FREESHIP trong 2 giờ' để tạo FOMO và thúc đẩy chốt đơn!")

### Bước 19: Tổng kết & đóng SparkSession

In [ ]:
print("\n" + "="*65)
print("  TỔNG KẾT BÀI TOÁN DỰ ĐOÁN KHẢ NĂNG MUA HÀNG")
print("="*65)
print(f"  Thuật toán tốt nhất : {best['model']}")
print(f"  ROC-AUC             : {best['roc_auc']:.4f}")
print(f"  F1-Score            : {best['f1']:.4f}  (↑ ưu tiên vì dữ liệu mất cân bằng)")
print(f"  MCC                 : {best['mcc']:.4f}  (↑ đáng tin cậy nhất)")
print(f"  Precision           : {best['precision']:.4f}")
print(f"  Recall              : {best['recall']:.4f}")
print("\n  Ba mô hình đã so sánh:")
print(f"  [LR]  F1={metrics_lr['f1']:.4f}  AUC={metrics_lr['roc_auc']:.4f}  MCC={metrics_lr['mcc']:.4f}")
print(f"  [RF]  F1={metrics_rf['f1']:.4f}  AUC={metrics_rf['roc_auc']:.4f}  MCC={metrics_rf['mcc']:.4f}")
print(f"  [GBT] F1={metrics_gbt['f1']:.4f}  AUC={metrics_gbt['roc_auc']:.4f}  MCC={metrics_gbt['mcc']:.4f}")
print("\n  Pipeline tiền xử lý:")
print("  StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler")
print("\n  Features sử dụng:")
print(f"  Numeric  : {numeric_cols}")
print(f"  Categorical (OHE): {cat_cols}")
print("\n  Ứng dụng:")
print("  Gửi mã FREESHIP tự động cho khách hàng P≥0.7 chưa chốt đơn")
print("="*65)

spark.stop()
print("\n SparkSession đã đóng.")